# Dense Retrieval

This notebook implements a dense retrieval system over the previously preprocessed course-material chunks. The `intfloat/multilingual-e5-base` model is used to transform questions and text chunks into dense vector representations, while a FAISS index enables efficient similarity search.

The retriever is evaluated on the validation set using Hit@k, Precision@k, Recall@k, F1@k, MRR@k, and nDCG@k for several values of `k`. The training, validation, and test retrieval results are saved in a uniform JSONL format so that the retrieved contexts can later be used by the Qwen3-8B and mT5 answer-generation experiments.

The required packages can be installed by running `%pip install sentence-transformers faiss-cpu pandas altair`. The embedding model is downloaded automatically from Hugging Face when it is loaded for the first time.

In [1]:
import json
from pathlib import Path

import altair as alt
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

## Dense Retrieval Configuration

In [2]:
RETRIEVER_NAME = "dense"
RETRIEVER_MODEL_NAME = "intfloat/multilingual-e5-base"
TOP_K_VALUES = (1, 3, 5, 10)
EXPORT_TOP_K = max(TOP_K_VALUES)

## Loading the Data

In [3]:
PROJECT_ROOT = Path.cwd()

CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "chunks_preprocessed.jsonl"
TRAIN_PATH = PROJECT_ROOT / "data" / "splits" / "train.jsonl"
VALIDATION_PATH = PROJECT_ROOT / "data" / "splits" / "validation.jsonl"
TEST_PATH = PROJECT_ROOT / "data" / "splits" / "test.jsonl"
RESULTS_DIR = PROJECT_ROOT / "data" / "retrieval"
ARTIFACTS_DIR = RESULTS_DIR / RETRIEVER_NAME


def load_jsonl(path: Path) -> list[dict]:
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Neispravan JSON u redu {line_number}: {path}"
                ) from error

    return records


chunks = load_jsonl(CHUNKS_PATH)
train_data = load_jsonl(TRAIN_PATH)
validation_data = load_jsonl(VALIDATION_PATH)
test_data = load_jsonl(TEST_PATH)

print(f"Broj chunkova: {len(chunks)}")
print(f"Training pitanja: {len(train_data)}")
print(f"Validation pitanja: {len(validation_data)}")
print(f"Test pitanja: {len(test_data)}")

Broj chunkova: 356
Training pitanja: 100
Validation pitanja: 21
Test pitanja: 22


In [4]:
required_chunk_fields = {
    "chunk_id", "processed_text", "pdf_page_start", "pdf_page_end"
}
required_question_fields = {
    "id", "processed_question", "processed_answer", "source_pages"
}

if not chunks:
    raise ValueError("Korpus chunkova je prazan.")

for chunk in chunks:
    missing = required_chunk_fields - chunk.keys()
    if missing:
        raise ValueError(f"Chunk {chunk.get('chunk_id')} nema polja: {sorted(missing)}")

for split_name, data in {
    "train": train_data,
    "validation": validation_data,
    "test": test_data,
}.items():
    for example in data:
        missing = required_question_fields - example.keys()
        if missing:
            raise ValueError(
                f"Pitanje {example.get('id')} iz skupa {split_name} nema polja: {sorted(missing)}"
            )

print("Provera strukture podataka je uspešna.")

Provera strukture podataka je uspešna.


## Model, Embeddings, and FAISS Index

The multilingual E5 model expects the `passage:` prefix for document chunks and the `query:` prefix for questions. These prefixes indicate the role of each input and help the model produce appropriate vector representations.

All embeddings are normalized before indexing. Therefore, the inner-product similarity calculated by FAISS `IndexFlatIP` is equivalent to cosine similarity. The resulting index enables efficient retrieval of the chunks that are semantically most similar to a given question.

In [5]:
retriever_model = SentenceTransformer(RETRIEVER_MODEL_NAME)

passages = [
    "passage: " + chunk["processed_text"].strip()
    for chunk in chunks
]

chunk_embeddings = retriever_model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

embedding_dimension = chunk_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dimension)
faiss_index.add(chunk_embeddings)

print(f"Oblik matrice embeddinga: {chunk_embeddings.shape}")
print(f"Broj vektora u indeksu: {faiss_index.ntotal}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Oblik matrice embeddinga: (356, 768)
Broj vektora u indeksu: 356


## Dense Retrieval Function

The following function transforms a question into a normalized query embedding and searches the FAISS index for the most semantically similar chunks. It returns the top-ranked chunks together with their similarity scores, page ranges, and associated text.

In [6]:
def retrieve_chunks(question: str, top_k: int = 5) -> list[dict]:
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Pitanje ne sme biti prazno.")
    if top_k < 1:
        raise ValueError("top_k mora biti pozitivan ceo broj.")

    query_embedding = retriever_model.encode(
        ["query: " + question.strip()],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    number_of_results = min(top_k, len(chunks))
    scores, indices = faiss_index.search(query_embedding, number_of_results)

    results = []
    for rank, (score, chunk_index) in enumerate(
        zip(scores[0], indices[0]), start=1
    ):
        chunk_index = int(chunk_index)
        if chunk_index < 0:
            continue

        chunk = chunks[chunk_index]
        results.append({
            "rank": rank,
            "chunk_id": chunk["chunk_id"],
            "score": float(score),
            "processed_text": chunk["processed_text"],
            "pdf_page_start": chunk["pdf_page_start"],
            "pdf_page_end": chunk["pdf_page_end"],
        })

    return results

In [7]:
example = validation_data[0]
question = example["processed_question"]
retrieved_chunks = retrieve_chunks(question=question, top_k=5)

print(f"ID pitanja: {example['id']}")
print(f"Pitanje: {question}")
print(f"Referentni odgovor: {example['processed_answer']}")
print(f"Referentne strane: {example['source_pages']}")

pd.DataFrame(retrieved_chunks)[
    ["rank", "chunk_id", "score", "pdf_page_start", "pdf_page_end"]
]

ID pitanja: 132
Pitanje: Šta je Cachegrind, za šta služi i kako se koristi?
Referentni odgovor: Cachegrind je Valgrind alat za profilisanje keš memorije. Koristi se pokretanjem programa kroz Cachegrind, nakon čega se analiziraju prikupljene informacije o pristupima kešu i izvršavanju.
Referentne strane: [194, 195, 196, 202]


,rank,chunk_id,score,pdf_page_start,pdf_page_end
0,1,chunk_0339,0.843073,195,196
1,2,chunk_0336,0.828881,194,195
2,3,chunk_0340,0.827606,196,197
3,4,chunk_0338,0.824414,195,196
4,5,chunk_0302,0.824103,176,177


## Evaluation Metrics

A retrieved chunk is considered relevant if its PDF page range contains at least one of the reference pages associated with the question. Retrieval performance is evaluated using Hit@k, Precision@k, Recall@k, F1@k, MRR@k, and nDCG@k.

In [8]:
def get_relevant_chunk_ids(
    gold_pages: list[int],
    chunks: list[dict],
) -> set[str]:
    return {
        chunk["chunk_id"]
        for chunk in chunks
        if any(
            chunk["pdf_page_start"] <= page <= chunk["pdf_page_end"]
            for page in gold_pages
        )
    }


def calculate_metrics_at_k(
    retrieved_ids: list[str],
    relevant_ids: set[str],
    k: int,
) -> dict[str, float]:
    top_k_ids = retrieved_ids[:k]
    hit_count = sum(chunk_id in relevant_ids for chunk_id in top_k_ids)

    hit = float(hit_count > 0)
    precision = hit_count / len(top_k_ids) if top_k_ids else 0.0
    recall = hit_count / len(relevant_ids) if relevant_ids else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    reciprocal_rank = 0.0
    for rank, chunk_id in enumerate(top_k_ids, start=1):
        if chunk_id in relevant_ids:
            reciprocal_rank = 1.0 / rank
            break

    dcg = sum(
        1.0 / np.log2(rank + 1)
        for rank, chunk_id in enumerate(top_k_ids, start=1)
        if chunk_id in relevant_ids
    )
    ideal_hit_count = min(len(relevant_ids), len(top_k_ids))
    idcg = sum(
        1.0 / np.log2(rank + 1)
        for rank in range(1, ideal_hit_count + 1)
    )
    ndcg = dcg / idcg if idcg > 0 else 0.0

    return {
        "Hit": hit,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "MRR": reciprocal_rank,
        "nDCG": ndcg,
    }

In [9]:
def evaluate_retriever(
    data: list[dict],
    chunks: list[dict],
    top_k_values: tuple[int, ...] = TOP_K_VALUES,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    max_k = max(top_k_values)
    per_query_rows = []

    for example in data:
        relevant_ids = get_relevant_chunk_ids(
            gold_pages=example["source_pages"],
            chunks=chunks,
        )
        if not relevant_ids:
            print(f"Preskočeno pitanje ID {example['id']}: nema relevantnih chunkova.")
            continue

        retrieved = retrieve_chunks(
            question=example["processed_question"],
            top_k=max_k,
        )
        retrieved_ids = [result["chunk_id"] for result in retrieved]

        for k in top_k_values:
            metrics = calculate_metrics_at_k(retrieved_ids, relevant_ids, k)
            per_query_rows.append({
                "question_id": example["id"],
                "k": k,
                "num_relevant": len(relevant_ids),
                **metrics,
            })

    if not per_query_rows:
        raise ValueError("Nijedno pitanje nije moglo da se evaluira.")

    per_query_df = pd.DataFrame(per_query_rows)
    metrics_df = (
        per_query_df
        .groupby("k", as_index=False)[
            ["Hit", "Precision", "Recall", "F1", "MRR", "nDCG"]
        ]
        .mean()
    )

    return metrics_df, per_query_df

## Evaluation on the Validation Set


In [10]:
dense_metrics_df, dense_per_query_metrics_df = evaluate_retriever(
    data=validation_data,
    chunks=chunks,
)

dense_metrics_df.round(6)

,k,Hit,Precision,Recall,F1,MRR,nDCG
0,1,0.714286,0.714286,0.156767,0.252646,0.714286,0.714286
1,3,0.714286,0.460317,0.289688,0.345733,0.714286,0.515032
2,5,0.809524,0.380952,0.400026,0.376060,0.735714,0.485820
3,10,0.952381,0.261905,0.539028,0.342780,0.754422,0.533136


In [11]:
metrics_long_df = dense_metrics_df.melt(
    id_vars="k",
    value_vars=["Hit", "Precision", "Recall", "F1", "MRR", "nDCG"],
    var_name="metric",
    value_name="value",
)

alt.Chart(metrics_long_df).mark_line(point=True).encode(
    x=alt.X("k:O", title="k"),
    y=alt.Y("value:Q", title="Prosečna vrednost", scale=alt.Scale(domain=[0, 1])),
    color=alt.Color("metric:N", title="Metrika"),
    tooltip=["metric:N", "k:O", alt.Tooltip("value:Q", format=".4f")],
).properties(
    width=650,
    height=350,
    title="Dense retrieval na validation skupu",
)

alt.Chart(...)

### **Validation Results Summary**

The dense retriever was evaluated on 21 validation questions. Hit@1 is 0.7143, which means that a relevant chunk was ranked first for approximately 71% of the questions. Hit@5 increases to 0.8095, while Hit@10 reaches 0.9524, indicating that at least one relevant chunk was retrieved among the top 10 results for approximately 95% of the validation questions.

Recall increases from 0.1568 at `k=1` to 0.5390 at `k=10`, since larger result sets contain more of the relevant chunks. At the same time, Precision decreases from 0.7143 to 0.2619 because retrieving more chunks also introduces additional non-relevant content.

The highest F1 score is obtained at `k=5`, with a value of 0.3761, which represents the best observed balance between Precision and Recall. The MRR values remain relatively high, indicating that the first relevant chunk is generally positioned near the top of the ranking.

Based on these results, the top 10 chunks are saved to provide flexibility for subsequent experiments, while the answer-generation models use the first 5 chunks as context.

## Selecting the Final Retrieval Configuration

The value of `k` is selected exclusively using the validation results. The
highest validation F1 score is obtained at `k=5`, so this value is selected as
the final dense-retrieval configuration.

The selected value is fixed before the test-set evaluation. Test results are
used only for the final evaluation and not for additional parameter tuning.

In [12]:
best_validation_row = dense_metrics_df.loc[
    dense_metrics_df["F1"].idxmax()
]

FINAL_TOP_K = int(
    best_validation_row["k"]
)

print(
    f"Izabrani finalni k: {FINAL_TOP_K}"
)

print(
    f"Validation F1@{FINAL_TOP_K}: "
    f"{best_validation_row['F1']:.6f}"
)

Izabrani finalni k: 5
Validation F1@5: 0.376060


## Final Evaluation on the Test Set

The dense retriever is evaluated on the held-out test set using the value
`k=5`, which was selected based on the validation F1 score. No parameters are
changed based on the test results.

In [13]:
FINAL_TOP_K = 5
dense_test_metrics_df, dense_test_per_query_metrics_df = (
    evaluate_retriever(
        data=test_data,
        chunks=chunks,
        top_k_values=(FINAL_TOP_K,),
    )
)

dense_test_metrics_df.round(6)

,k,Hit,Precision,Recall,F1,MRR,nDCG
0,5,0.681818,0.272727,0.285227,0.27219,0.543182,0.336171


## Saving the Retrieval Results

For each data split, the question, reference answer, source pages, and ranked chunks are saved in JSONL format. These results can be loaded directly by the Qwen3-8B and `google/mt5-base` answer-generation notebooks, without running the dense retriever again.

In [14]:
def save_jsonl(records: list[dict], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")


def build_retrieval_records(
    data: list[dict],
    split_name: str,
    top_k: int = EXPORT_TOP_K,
) -> list[dict]:
    records = []

    for example in data:
        retrieved = retrieve_chunks(
            question=example["processed_question"],
            top_k=top_k,
        )

        records.append({
            "question_id": example["id"],
            "question": example["question"],
            "processed_question": example["processed_question"],
            "lexical_question": example.get("lexical_question"),
            "answer": example["answer"],
            "source_pages": example["source_pages"],
            "retrieved_chunks": retrieved,
        })

    return records

In [15]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
np.save(ARTIFACTS_DIR / "dense_embeddings.npy", chunk_embeddings)
faiss.write_index(faiss_index, str(ARTIFACTS_DIR / "dense_index.faiss"))

metadata = {
    "model_name": RETRIEVER_MODEL_NAME,
    "passage_input": "processed_text",
    "question_input": "processed_question",
    "query_prefix": "query: ",
    "passage_prefix": "passage: ",
    "top_k": EXPORT_TOP_K,
    "n_chunks": len(chunks),
    "embedding_dimension": embedding_dimension,
}
with (ARTIFACTS_DIR / "dense_metadata.json").open("w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

retrieval_splits = {
    "train": train_data,
    "validation": validation_data,
    "test": test_data,
}

for split_name, split_data in retrieval_splits.items():
    records = build_retrieval_records(
        data=split_data,
        split_name=split_name,
    )
    output_path = RESULTS_DIR / f"dense_{split_name}_top{EXPORT_TOP_K}.jsonl"
    save_jsonl(records, output_path)
    print(f"Sačuvano {len(records)} primera: {output_path}")

dense_metrics_df.to_csv(
    ARTIFACTS_DIR / "validation_metrics.csv",
    index=False,
)
dense_per_query_metrics_df.to_csv(
    ARTIFACTS_DIR / "validation_per_query_metrics.csv",
    index=False,
)

dense_test_metrics_df.to_csv(
    ARTIFACTS_DIR / "test_metrics.csv",
    index=False,
)

dense_test_per_query_metrics_df.to_csv(
    ARTIFACTS_DIR / "test_per_query_metrics.csv",
    index=False,
)

print("Sačuvane su i validation metrike.")

Sačuvano 100 primera: /home/jelena/Desktop/faks/masinsko/Student-Question-Answering-from-Course-Materials/data/retrieval/dense_train_top10.jsonl
Sačuvano 21 primera: /home/jelena/Desktop/faks/masinsko/Student-Question-Answering-from-Course-Materials/data/retrieval/dense_validation_top10.jsonl
Sačuvano 22 primera: /home/jelena/Desktop/faks/masinsko/Student-Question-Answering-from-Course-Materials/data/retrieval/dense_test_top10.jsonl
Sačuvane su i validation metrike.
